### 복습 (데이터프레임)
1. 민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json 파일을 로드
2. 고객질문(요청), 상담사답변 컬럼의 문자 정규화 (특수문자 제거, 공백제거)
3. 고객질문에 대한 즉각적인 상담사의 답변이 있는 행들만 남기고 나머지는 제거
4. 고객의 질문과 상담사의 답변을 하나의 행으로 결합
5. 인덱스를 초기화하고 label 컬럼을 생성하여 1을 대입

In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
df = pd.read_json("data/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50336 entries, 0 to 50335
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   도메인        50336 non-null  object
 1   카테고리       50336 non-null  object
 2   대화셋일련번호    50336 non-null  object
 3   화자         50336 non-null  object
 4   문장번호       50336 non-null  int64 
 5   고객의도       50336 non-null  object
 6   상담사의도      50336 non-null  object
 7   QA         50336 non-null  object
 8   고객질문(요청)   50336 non-null  object
 9   상담사질문(요청)  50336 non-null  object
 10  고객답변       50336 non-null  object
 11  상담사답변      50336 non-null  object
 12  개체명        50336 non-null  object
 13  용어사전       50336 non-null  object
 14  지식베이스      50336 non-null  object
dtypes: int64(1), object(14)
memory usage: 5.8+ MB


In [4]:
# 컬럼의 이름에서 좌우의 공백을 제거 (strip() -> str 내장함수)
df.columns.map( lambda x : x.strip() )
# [x.strip() for x in df.columns] <- 이것도 같은 기능

Index(['도메인', '카테고리', '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA',
       '고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변', '개체명', '용어사전', '지식베이스'],
      dtype='object')

In [5]:
# 정규화
def normalize_token_text(text : str) -> str:
    text = re.sub(r'[^가-힣a-zA-Z0-9\s\.]', " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
df['고객질문(요청)'] = df['고객질문(요청)'].map(normalize_token_text)
df['상담사답변'] = df['상담사답변'].map(normalize_token_text)

In [6]:
df.head()

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까,,,,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
2,다산콜센터,일반행정 문의,B2240,고객,3,지방세납부,,Q,은행 어플에서도 됩니까,,,,"은행, 어플",은행/공공기관,"어플,공공기관"
3,다산콜센터,일반행정 문의,B2240,상담사,4,,지방세납부,Q,,어떤 은행을 이용하고 계십니까?,,,은행,은행/공공기관,"은행,공공기관"
4,다산콜센터,일반행정 문의,B2240,고객,5,지방세납부,,A,,,기업은행을 이용하고 있습니다.,,기업은행,기업은행/상호,"기업은행,상호"


In [7]:
# 고객질문-답변 행 남기고 나머지 제거

# 1번 조건식 -> 현재 행에서 고객질문(요청) 데이터가 ''가 아니고 다음행의 상담사답변의 value가 ''이 아닌 경우
flag1 = (df['고객질문(요청)'] != '') & (df['상담사답변'].shift(-1) != '')
# 2번 조건식 -> 현재 행에서 상담사 답변이 ''가 아니고 전 행의 고객질문 데이터가 ''가 아닌 경우
flag2 = (df['상담사답변'] != '') & (df['고객질문(요청)'] != '' )


# 질문-답변의 인덱스 맞추기
df['상담사답변'] = df['상담사답변'].shift(-1)

# 고객질문 데이터에서 ''가 아닌 데이터만 필터
df = df.loc[
    df['고객질문(요청)'] != ''
]
df.drop_duplicates('고객질문(요청)')

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,지방세,지방세/세금,"지방세,세금"
2,다산콜센터,일반행정 문의,B2240,고객,3,지방세납부,,Q,은행 어플에서도 됩니까,,,,"은행, 어플",은행/공공기관,"어플,공공기관"
6,다산콜센터,일반행정 문의,B2240,고객,7,지방세납부,,Q,은행을 직접방문해도 됩니까,,,방문납부도 가능합니다.,은행,은행/공공기관,"은행,공공기관"
8,다산콜센터,일반행정 문의,B2240,고객,9,지방세납부,,Q,은행위치 좀 알 수 있습니까,,,,은행,은행/공공기관,"은행,공공기관"
12,다산콜센터,일반행정 문의,B2240,고객,13,지방세납부,,Q,버스로 가는 방법도 있습니까,,,도보로 가시는게 빠를 것 같습니다.,버스,버스/교통수단,"버스,교통수단"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50324,다산콜센터,일반행정 문의,B35809,고객,9,여성전용아파트,,Q,꼭 서울시에 근무해야하나요,,,서울소재 직장근무로 제한하고있습니다.,"서울시, 근무","서울시/서울/서울특별시, 근무",근무
50326,다산콜센터,일반행정 문의,B35809,고객,11,여성전용아파트,,Q,임대료는 어떻게 되나요,,,62400원 입니다.,임대료,"임대료, 월세","임대료,월세"
50328,다산콜센터,일반행정 문의,B35809,고객,13,여성전용아파트,,Q,보증금도 있나요,,,1423200원 입니다.,보증금,보증금/예치금,"보증금,예치금"
50332,다산콜센터,일반행정 문의,B35809,고객,17,여성전용아파트,,Q,입주순위도 있나요,,,1 3순위가 있습니다.,"입주, 순위","입주/이사, 순위/순서","순위,순서"


In [8]:
# 인덱스 초기화
df.reset_index(inplace=True)

In [9]:
# label 컬럼을 생성하여 1을 대입
df['label'] = 1
# 해당하는 행의 질문과 답변은 정상적인 답변입니다.

In [10]:
# 상담사 답변이 비어있는 행 제거
df = df[df['상담사답변'].str.strip() != '']
df = df.reset_index(drop=True)

In [11]:
# 컬럼 이름 바꾸기
df.rename(columns={
    '고객질문(요청)' : '고객질문'
}, inplace=True)

In [12]:
df.head()

,index,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문,상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스,label
0,0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,지방세,지방세/세금,"지방세,세금",1
1,6,다산콜센터,일반행정 문의,B2240,고객,7,지방세납부,,Q,은행을 직접방문해도 됩니까,,,방문납부도 가능합니다.,은행,은행/공공기관,"은행,공공기관",1
2,12,다산콜센터,일반행정 문의,B2240,고객,13,지방세납부,,Q,버스로 가는 방법도 있습니까,,,도보로 가시는게 빠를 것 같습니다.,버스,버스/교통수단,"버스,교통수단",1
3,14,다산콜센터,일반행정 문의,B2240,고객,15,지방세납부,,Q,다른 납부방법도 있습니까,,,위택스 사이트에서 납부하실수 있습니다.,,납부방법/지방세,,1
4,16,다산콜센터,일반행정 문의,B2240,고객,17,지방세납부,,Q,사이트 주소가 어떻게 됩니까,,,www.wetax.go.kr 입니다.,사이트,납부방법/지방세/ 위텍스/ 납부,"사이트,납부",1


In [13]:
# label이 0인 데이터들을 생성 -> 기존의 질문과 답변에서 질문은 그대로 유지한채
# 원래의 답변을 제외한 다른 랜덤한 답으로 채운다
answers_list = df[['고객질문', '상담사답변']].values.tolist()
answers_list

[['지방세를 내려면 어떻게 해야됩니까', '이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.'],
 ['은행을 직접방문해도 됩니까', '방문납부도 가능합니다.'],
 ['버스로 가는 방법도 있습니까', '도보로 가시는게 빠를 것 같습니다.'],
 ['다른 납부방법도 있습니까', '위택스 사이트에서 납부하실수 있습니다.'],
 ['사이트 주소가 어떻게 됩니까', 'www.wetax.go.kr 입니다.'],
 ['다른곳에서는 납부할수 없습니까', '이용하시는 은행의 사이트에서도 지방세 납부가 가능합니다.'],
 ['지방세는 조회할 수 있습니까', '간단한 본인확인 후 안내해드리겠습니다.'],
 ['어떻게 납부합니까', '온라인으로는 위택스나 은행 홈페이지에서 가능합니다.'],
 ['사이트 주소가 어떻게 됩니까', 'OOOO.OOO.go.kr 입니다.'],
 ['서울시주최 페스티벌 예매해놨는데 예정대로 진행됩니까', '현재로썬 진행될 예정입니다.'],
 ['코로나로 다른 축제들은 취소됐는데 이건 취소 안됩니까', '네 철저한 방역수칙하에 진행할 예정입니다.'],
 ['환불할수 있습니까', '예매사이트 통해서 환불하실 수 있습니다.'],
 ['환불수수료 있습니까', '공연 일주일 전까진 10 수수료가 있습니다.'],
 ['코로나때문에 못가는건데도 수수료 내야합니까', '네 규정상 어쩔수 없습니다.'],
 ['참석했다가 코로나 걸리면 보상해줍니까', '철저한 방역수칙으로 진행하기에 걱정 안하셔도 됩니다.'],
 ['그래도 걸리면 보상해줍니까', '그 부분에 대해서는 개인이 조심을 하셔야기때문에 서울시에서는 따로 보상을 해드리진 않습니다.'],
 ['확산세가 더 강해지면 취소될 가능성 있습니까', '확산세가 심하면 취소 될 수도 있습니다.'],
 ['그럼 전액 환불되는겁니까', '네 맞습니다.'],
 ['청년저축계좌 지금 신청할 수 있습니까', '죄송하지만 이미 신청기간이 지났습니다.'],
 ['그럼 다른 정책은 없습니까', '희망두배 청년통장은 신청하실수 있습니

In [14]:
# 질문은 유지한채 답변을 다른 답변으로 변경하여 새로운 2차원 리스트 생성
neg_list = []
for q, a in answers_list:
    # q -> 질문
    # a -> 답변
    # 반복문에서 정상적인 답변을 제외한 나머지 답변의 리스트 생성
    cand = [ a2 for q2, a2 in answers_list if a2 != a]
    # 틀린 답변 하나를 선택
    neg_a = np.random.choice(cand)
    neg_list.append([q, neg_a] )

In [15]:
answers_list2 = [
    ['A', 'a'],
    ['B', 'b'],
    ['C', 'c'],
    ['D', 'd'],
    ['E', 'e']
]
neg_list2 = []
for q, a in answers_list2:
    # 1회차 반복 ->q('A'), a('a')
    # cand = [ a2 for q2, a2 in answers_list2 if a2 != a]
    cand = []
    for q2, a2 in answers_list2:
        if a2 != a:
            cand.append(a2)
    
    neg_a = np.random.choice(cand)
    neg_list2.append([q, neg_a] )

In [16]:
neg_list2

[['A', np.str_('e')],
 ['B', np.str_('a')],
 ['C', np.str_('d')],
 ['D', np.str_('a')],
 ['E', np.str_('d')]]

In [17]:
neg_df = pd.DataFrame(neg_list, columns = ['고객질문', '상담사답변'])
neg_df.head()

,고객질문,상담사답변
0,지방세를 내려면 어떻게 해야됩니까,네 원하시는 횟수 만큼 분할해서 납부하실 수도 있습니다.
1,은행을 직접방문해도 됩니까,네 안녕하세요.
2,버스로 가는 방법도 있습니까,위택스라고 검색하셔서 자동차세 연납 신청하시면 됩니다.
3,다른 납부방법도 있습니까,400번 502번 버스를 이용하시면 됩니다.
4,사이트 주소가 어떻게 됩니까,방문접수 또는 우편접수로 받고 있습니다.


In [18]:
neg_df['label'] = 0

In [19]:
# df와 neg_df를 단순한 행 결합
dataset_df = pd.concat([df.head(1000), neg_df.head(1000)], axis=0, ignore_index=True)

In [20]:
dataset_df['label'].value_counts()

label
1    1000
0    1000
Name: count, dtype: int64

In [21]:
dataset_df

,index,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문,상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스,label
0,0.0,다산콜센터,일반행정 문의,B2240,고객,1.0,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,지방세,지방세/세금,"지방세,세금",1
1,6.0,다산콜센터,일반행정 문의,B2240,고객,7.0,지방세납부,,Q,은행을 직접방문해도 됩니까,,,방문납부도 가능합니다.,은행,은행/공공기관,"은행,공공기관",1
2,12.0,다산콜센터,일반행정 문의,B2240,고객,13.0,지방세납부,,Q,버스로 가는 방법도 있습니까,,,도보로 가시는게 빠를 것 같습니다.,버스,버스/교통수단,"버스,교통수단",1
3,14.0,다산콜센터,일반행정 문의,B2240,고객,15.0,지방세납부,,Q,다른 납부방법도 있습니까,,,위택스 사이트에서 납부하실수 있습니다.,,납부방법/지방세,,1
4,16.0,다산콜센터,일반행정 문의,B2240,고객,17.0,지방세납부,,Q,사이트 주소가 어떻게 됩니까,,,www.wetax.go.kr 입니다.,사이트,납부방법/지방세/ 위텍스/ 납부,"사이트,납부",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1시간권 말고 또 있나요,NaN,NaN,복사는 하드디스크에서 이동식USB를 복사하는 방법이 있다고 합니다.,NaN,NaN,NaN,0
1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,시간이 넘어가면 추가금액이 있어요,NaN,NaN,무고한 피해를 줄이기 위한 제도로 알고 있습니다.,NaN,NaN,NaN,0
1997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,아 그것도 따로 있어요,NaN,NaN,네. 그렇습니다.,NaN,NaN,NaN,0
1998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,한시간권 30일 이용은 얼마인가요,NaN,NaN,네 가능합니다,NaN,NaN,NaN,0


In [22]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
# train, test 셋으로 데이터프레임 분할
train_df, test_df = train_test_split(
    dataset_df, test_size=0.2, random_state=42, stratify=dataset_df['label']
)

In [24]:
train_df['label'].value_counts()

label
1    800
0    800
Name: count, dtype: int64

In [25]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))
ds = DatasetDict(
    {
        'train' : train_ds, 
        'test' : test_ds
    }
)
ds

DatasetDict({
    train: Dataset({
        features: ['index', '도메인', '카테고리', '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA', '고객질문', '상담사질문(요청)', '고객답변', '상담사답변', '개체명 ', '용어사전', '지식베이스', 'label'],
        num_rows: 1600
    })
    test: Dataset({
        features: ['index', '도메인', '카테고리', '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA', '고객질문', '상담사질문(요청)', '고객답변', '상담사답변', '개체명 ', '용어사전', '지식베이스', 'label'],
        num_rows: 400
    })
})

In [26]:
import torch
from transformers import AutoTokenizer, DataCollatorWithPadding
# BertForSequenceClassification : BertModel -> <CLS> 토큰벡터 추출 -> dropout -> Linear까지 작업
from transformers import Trainer, TrainingArguments, BertForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

In [27]:
# 토큰화
MODEL_NAME = 'skt/kobert-base-v1'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast = False)
max_len = 256       # 장문의 데이터인 경우에는 512까지 조절

def tok_fn(batch):
    # token화 할 데이터가 2개의 컬럼에 나눠져있다.
    enc = tokenizer(
        batch['고객질문'],
        batch['상담사답변'],
        truncation = True,
        max_length = max_len
    )
    # 일반적인 kobert 모델에서는 token_type_ids의 데이터 영역 필요 X
    enc.pop("token_type_ids", None) # 완전히 제거
    return enc

# remove_columns -> 특정 컬럼의 데이터를 제외 -> label컬럼을 제외한 나머지 모두
tok_ds = ds.map(tok_fn, batched = True,
                remove_columns=[col for col in dataset_df.columns if col not in ['label']],
                # 캐시 사용 안 함
                load_from_cache_file=False,
                desc = 'tokenizer clean'
                )

tokenizer clean: 100%|██████████| 400/400 [00:00<00:00, 17004.57 examples/s]


In [28]:
tok_ds

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1600
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 400
    })
})

In [29]:
# 배치마다 동적으로 padding토큰을 추가
collator = DataCollatorWithPadding(
    tokenizer = tokenizer
)

In [39]:
# BertModel -> CLS -> dropout -> linear
model = BertForSequenceClassification.from_pretrained(MODEL_NAME,
                                                      num_labels = 2)
 

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at skt/kobert-base-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [40]:
# 평가지표 함수
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis = -1)
    return {
        'accuracy_score' : accuracy_score(labels, preds),
        'f1_score' : f1_score(labels, preds)
    }

In [41]:
args = TrainingArguments(
    output_dir='./kobert_pair_cls',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    num_train_epochs=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    greater_is_better=True,
    report_to=[]
)

In [42]:
trainer = Trainer(
    model = model,
    args = args,
    train_dataset=tok_ds['train'],
    eval_dataset = tok_ds['test'],
    tokenizer = tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

C:\Users\ekfla\AppData\Local\Temp\ipykernel_31720\2085847498.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [43]:
trainer.train()

c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy Score,F1 Score
1,0.567100,0.552255,0.697500,0.705596
2,0.477100,0.678540,0.662500,0.742857
3,0.423900,0.616258,0.757500,0.782022


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=600, training_loss=0.5305678049723307, metrics={'train_runtime': 328.6694, 'train_samples_per_second': 14.604, 'train_steps_per_second': 1.826, 'total_flos': 99250423414080.0, 'train_loss': 0.5305678049723307, 'epoch': 3.0})

In [44]:
# 평가
def score_pairs(question, answers):
    # 질문 1개 -> 답변 여러개 
    questions = [question] * len(answers)
    enc = tokenizer(
        questions,
        answers,
        return_tensors = 'pt',
        truncation = True,
        max_length = max_len,
        padding = True
    )
    # token_type_ids 제거
    enc.pop("token_type_ids", None)

    # 인코딩 데이터를 딕셔너리 형태로 변환
    inputs = {
        k : v for k, v in enc.items()
    }

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax( logits, dim = -1 )[:,1]  # label이 1일 확률
        # probs가 클수록 올바른 답변
    return probs.cpu().tolist()

In [45]:
question = '지방세 납부는 어디서 할수 있을까요?'
answers = [
    '이용하시는 은행 사이트나 앱에서 지방세 납부가 가능합니다',
    '지방층이 너무 두껍습니다',
    '동물 등록은 거주지 구청에서 처리하셔야 합니다.',
    '출입국 관련 업무는 외교부에서 담당합니다.'
]
scores = score_pairs(question, answers)

In [46]:
scores

[0.07260745018720627,
 0.08065266162157059,
 0.12948253750801086,
 0.134985089302063]

In [47]:
best_idx = int(np.argmax(scores))
for i, (a, s) in enumerate(zip(answers, scores)):
    # i -> 답변
    # a -> 답변
    # s -> 1의 적합 확률
    print(f"{i} {a} -> 적합확률 : {round(s, 3)}")
print(f'적합한 답변은 {best_idx} : {answers[best_idx]}')


0 이용하시는 은행 사이트나 앱에서 지방세 납부가 가능합니다 -> 적합확률 : 0.073
1 지방층이 너무 두껍습니다 -> 적합확률 : 0.081
2 동물 등록은 거주지 구청에서 처리하셔야 합니다. -> 적합확률 : 0.129
3 출입국 관련 업무는 외교부에서 담당합니다. -> 적합확률 : 0.135
적합한 답변은 3 : 출입국 관련 업무는 외교부에서 담당합니다.
